## Exercițiul 1: Profil de utilizator — combinarea a 3 resurse API (JSONPlaceholder)

Avem 3 resurse separate de la [JSONPlaceholder](https://jsonplaceholder.typicode.com/), legate prin foreign keys (`userId`, `postId`): `users`, `posts`, `comments`.

**Cerințe:**
1. Construiți mai întâi un `dict` `{postId: nr_comentarii}` din `comments`.
2. Pentru fiecare utilizator, construiți o structură **imbricată**:
   ```python
   {
       "nume": ...,
       "nr_postari": ...,
       "total_comentarii": ...,   # suma comentariilor la toate postarile lui
       "postari": [{"titlu": ..., "nr_comentarii": ...}, ...],
   }
   ```
   într-un `dict` `profil_utilizatori`, cheia fiind `username`.
3. Găsiți utilizatorul cu cele mai multe comentarii primite în total.
4. Bonus: sortați `profil_utilizatori` descrescător după `total_comentarii` (`sorted()` cu `key`).

In [ ]:
import requests

users = requests.get("https://jsonplaceholder.typicode.com/users").json()
posts = requests.get("https://jsonplaceholder.typicode.com/posts").json()
comments = requests.get("https://jsonplaceholder.typicode.com/comments").json()

print("users:", len(users), "| posts:", len(posts), "| comments:", len(comments))
print("\nexemplu user:", users[0]["username"], users[0]["name"])
print("exemplu post:", {"userId": posts[0]["userId"], "id": posts[0]["id"], "titlu": posts[0]["title"]})
print("exemplu comment:", {"postId": comments[0]["postId"]})


## Exercițiul 2: Prognoza pe ore, grupată pe zi (Open-Meteo)

Preluăm temperaturi orare pentru 3 zile. Câmpul `time` e un text de forma `"2026-09-14T13:00"` — primele 10 caractere sunt data.

**Cerințe:**
1. Grupați valorile orare într-un `dict` `{data: [temperaturi]}`, folosind `dict.setdefault()` și `data_text[:10]` ca și cheie.
2. Construiți un `dict` imbricat `rezumat_zilnic = {data: {"min": ..., "max": ..., "medie": ...}}`.
3. Găsiți ziua cu cea mai mare diferență `max - min` (cea mai "instabilă" zi).
4. Bonus: găsiți, per zi, **ora** la care a fost înregistrată temperatura maximă.

In [ ]:
import requests

raspuns = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": 44.43, "longitude": 26.10, "hourly": "temperature_2m", "forecast_days": 3},
)
date_meteo = raspuns.json()

ore = date_meteo["hourly"]["time"]
temperaturi = date_meteo["hourly"]["temperature_2m"]

print("nr inregistrari orare:", len(ore))
print("exemplu:", ore[0], "->", temperaturi[0], "grade")


## Exercițiul 3: Cache + comparare echipe (PokeAPI)

Funcția `get_pokemon()` de mai jos ține un `dict` `cache_pokemoni` — dacă un pokemon a mai fost cerut, îl întoarce direct din cache, fără un nou apel API. Observați că `"gyarados"` apare în **ambele** echipe.
Cache = memorie care pastreaza datele ultimei accesari

**Cerințe:**
1. Construiți o funcție `tipuri_echipa(echipa)` care întoarce un `set` cu toate tipurile din acea echipă (un pokemon poate avea 1-2 tipuri).
2. Comparați `echipa_1` și `echipa_2`: ce tipuri au în comun? Ce tipuri are doar `echipa_1`?
3. Construiți o funcție `putere_totala(echipa)` care însumează toate statisticile de bază (`hp + attack + defense + ...`) ale echipei.
4. După ce ați rulat exercițiul, verificați `len(cache_pokemoni)` — de ce nu sunt 6 intrări, deși ambele echipe au câte 3 pokemoni?

In [ ]:
import requests

cache_pokemoni = {}

def get_pokemon(nume):
    if nume not in cache_pokemoni:
        raspuns = requests.get(f"https://pokeapi.co/api/v2/pokemon/{nume}")
        raspuns.raise_for_status()
        cache_pokemoni[nume] = raspuns.json()
    return cache_pokemoni[nume]

echipa_1 = ["pikachu", "charizard", "gyarados"]
echipa_2 = ["bulbasaur", "venusaur", "gyarados"]


## Exercițiul 4: Repo-urile unei organizații GitHub, grupate pe limbaj

Folosim [GitHub REST API](https://docs.github.com/en/rest) (public, fără autentificare pentru date publice). **Atenție**: câmpul `language` poate fi `None` pentru unele repo-uri (ex: `.github`, repo-uri doar cu configurări) — o valoare lipsă reală, de tratat explicit.
Repo = folder unde este salvat codul

**Cerințe:**
1. Grupați repo-urile într-un `dict` `{limbaj: [nume_repo]}`, tratând `None` ca `"Necunoscut"`.
2. Construiți un `dict` `{limbaj: total_stele}` — suma stelelor (`stargazers_count`) per limbaj.
3. Construiți un `dict` `{limbaj: {"nume": ..., "stele": ...}}` cu **cel mai popular** repo per limbaj.
4. Sortați limbajele descrescător după `total_stele` și afișați clasamentul.
5. Bonus: repetați exercițiul pentru altă organizație (ex. `"numpy"`, `"psf"`) și comparați cele două clasamente cu un `set` de limbaje comune.

In [1]:
import requests

raspuns = requests.get(
    "https://api.github.com/orgs/pandas-dev/repos",
    params={"per_page": 30},
)
repo_uri = raspuns.json()

print("nr repo-uri:", len(repo_uri))
for r in repo_uri[:5]:
    print(f"  {r['name']:25s} | limbaj={r['language']} | stele={r['stargazers_count']}")


nr repo-uri: 16
  pandas                    | limbaj=Python | stele=49734
  pandas-governance         | limbaj=None | stele=36
  pandas-msgpack            | limbaj=Python | stele=25
  pandas-compat             | limbaj=Python | stele=7
  pandas-release            | limbaj=Python | stele=8
